# Music Genre Classification Project


In [ ]:
!pip3 install numpy pandas matplotlib librosa scikit-learn tensorflow

In [2]:
import librosa
import numpy as np
import matplotlib.pyplot as plt
import os
import librosa.display

In [3]:
def save_spectrogram(audio_path, output_path):
    y, sr = librosa.load(audio_path, sr=22050)
    spectrogram = librosa.feature.melspectrogram(y=y, sr=sr)
    log_spectrogram = librosa.power_to_db(spectrogram)
    plt.figure(figsize=(3, 3))
    librosa.display.specshow(log_spectrogram, sr=sr, cmap='magma')
    plt.axis('off')
    plt.savefig(output_path, bbox_inches='tight', pad_inches=0)
    plt.close()

In [ ]:
data_dir = 'Data/genres_original/'
output_dir = 'Data/spectrograms/'
genres = os.listdir(data_dir)
genres = [g for g in os.listdir(data_dir) if not g.startswith('.')]
for genre in genres:
    genre_dir = os.path.join(data_dir, genre)
    output_genre_dir = os.path.join(output_dir, genre)
    os.makedirs(output_genre_dir, exist_ok=True)
    for file in os.listdir(genre_dir):
        if file.endswith('.wav'):
            audio_path = os.path.join(genre_dir, file)
            output_path = os.path.join(output_genre_dir, file.replace('.wav', '.png'))
            save_spectrogram(audio_path, output_path)

In [3]:
!pip3 install --upgrade soundfile



[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: python3.12 -m pip install --upgrade pip


In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
img_size = (128, 128)
batch_size = 32
datagen = ImageDataGenerator(validation_split=0.2, rescale=1./255)
train_data = datagen.flow_from_directory('Data/spectrograms/', target_size=img_size, batch_size=batch_size, class_mode='categorical', subset='training')
val_data = datagen.flow_from_directory('Data/spectrograms/', target_size=img_size, batch_size=batch_size, class_mode='categorical', subset='validation')

Found 800 images belonging to 11 classes.
Found 199 images belonging to 11 classes.


In [ ]:
from tensorflow.keras import layers, models
model = models.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(128,128,3)),
    layers.MaxPooling2D(2,2),
    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(11, activation='softmax')
])
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_2 (Conv2D)               │ (None, 126, 126, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 63, 63, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 61, 61, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 30, 30, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 57600)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │     7,372,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 11)             │         1,419 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,393,739 (28.20 MB)

 Trainable params: 7,393,739 (28.20 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history = model.fit(train_data, epochs=20, validation_data=val_data)

Epoch 1/20


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


25/25 ━━━━━━━━━━━━━━━━━━━━ 5s 152ms/step - accuracy: 0.1229 - loss: 3.8879 - val_accuracy: 0.3216 - val_loss: 2.0570
Epoch 2/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 4s 137ms/step - accuracy: 0.2527 - loss: 2.0777 - val_accuracy: 0.3920 - val_loss: 1.8608
Epoch 3/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 4s 153ms/step - accuracy: 0.3417 - loss: 1.7961 - val_accuracy: 0.3467 - val_loss: 1.8560
Epoch 4/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 5s 189ms/step - accuracy: 0.4161 - loss: 1.5879 - val_accuracy: 0.4422 - val_loss: 1.6958
Epoch 5/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 5s 166ms/step - accuracy: 0.4667 - loss: 1.4704 - val_accuracy: 0.4472 - val_loss: 1.6617
Epoch 6/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 4s 149ms/step - accuracy: 0.5599 - loss: 1.2611 - val_accuracy: 0.4121 - val_loss: 1.5873
Epoch 7/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 4s 147ms/step - accuracy: 0.6171 - loss: 1.1081 - val_accuracy: 0.4372 - val_loss: 1.4877
Epoch 8/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 4s 161ms/step - accuracy: 0.6645 - loss: 0.9657 - val_accuracy: 0.4322 - val_

In [ ]:
model.evaluate(val_data)

7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step - accuracy: 0.4957 - loss: 1.7239


[1.7098764181137085, 0.5326633453369141]

In [ ]:
def predict_genre(audio_file):
    temp_path = 'temp_spec.png'
    save_spectrogram(audio_file, temp_path)
    from tensorflow.keras.preprocessing import image
    img = image.load_img(temp_path, target_size=img_size)
    img_array = image.img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)
    prediction = model.predict(img_array)
    predicted_index = np.argmax(prediction)
    predicted_genre = list(train_data.class_indices.keys())[predicted_index]
    os.remove(temp_path)
    return predicted_genre


In [ ]:
print(predict_genre('/Users/utkarshgaur/python3/project audio/test sample/jazz-cafe-crowd-319309.wav'))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 149ms/step
jazz
